# Grover Subset Sum Oracle Following the Paper Outline

This notebook implements a small subset-sum oracle in the style described in:

**Angelo Benoit, Sam Schwartz, Ron K. Cytron,  
"Optimization of a Quantum Subset Sum Oracle", arXiv:2410.01775.**

The goal is **not** to copy the authors' repository and **not** to hide the arithmetic inside `WeightedAdder`.

Instead, this notebook follows the written oracle outline:

1. selector register x, where x_i=1 means include a_i,
2. phase-kickback ancilla y=|->,
3. constant registers a_i,
4. shadow registers b_i, with b_i^j = a_i^j AND x_i,
5. running sums of the shadow registers,
6. equality comparison with target T,
7. phase kickback,
8. uncomputation.

For a small demo, we use:

S = {1, 2, 3}, T = 3.

The valid subsets are {3} and {1, 2}.

## What is the adder, and why is it the issue?

The oracle must compute

Z(x) = sum_i x_i a_i.

This is not a classical computation outside the circuit. It must be done **reversibly** on quantum registers.

Classically, if you write

```python
z = z + a
```

you overwrite information. Quantum circuits must be unitary, so the operation must be reversible, for example:

|a>|z> -> |a>|z+a>.

That is what a **quantum adder** does.

The paper discusses addition bit-by-bit using sum and carry logic. The hard part is that carry propagation costs gates and ancillas. For large subset-sum instances, the adder dominates the qubit and gate cost. This is why the paper's real contribution is mostly **oracle optimization**, not a new Grover algorithm.

In this notebook, to keep the circuit transparent, we implement addition using controlled increments:

|c>|s> -> |c>|s + c 2^j>.

Then adding a register b to a sum register is done by applying one controlled increment for each bit b_j. This is not the most optimized adder, but it is explicit and follows the paper's spirit.

In [ ]:
# If needed:
# !pip install qiskit qiskit-aer matplotlib pylatexenc

from math import pi, sqrt, floor
from itertools import product

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator

In [ ]:
# Problem instance

S = [1, 2, 3]
T = 3
n = len(S)

def bit_width(m: int) -> int:
    '''Number of bits needed for a nonnegative integer.'''
    return max(1, m.bit_length())

W = bit_width(sum(S))

print("S =", S)
print("T =", T)
print("n =", n)
print("bit width W =", W)

# Classical check, only to verify the quantum output.
# This is not used to build the oracle.
solutions = []
for bits in product([0, 1], repeat=n):
    if sum(bit * a for bit, a in zip(bits, S)) == T:
        solutions.append(bits)

print("Classical solution bitstrings, in x[0],x[1],x[2] order:")
print(solutions)

## Helper functions

We use little-endian registers:

reg[0] = least significant bit.

So the integer 3, with width 3, is represented as bits 011, but in little-endian register order this is [1,1,0].

In [ ]:
def little_endian_bits(value: int, width: int):
    '''Return [bit0, bit1, ..., bit_{width-1}].'''
    return [(value >> j) & 1 for j in range(width)]


def initialize_constant_register(qc: QuantumCircuit, reg, value: int):
    '''
    Initialize a register to a classical integer value.
    Little-endian convention: reg[0] is LSB.
    '''
    for j, bit in enumerate(little_endian_bits(value, len(reg))):
        if bit == 1:
            qc.x(reg[j])


def build_shadow_register(qc: QuantumCircuit, x_i, a_reg, b_reg):
    '''
    Build b_i^j = a_i^j AND x_i.

    This matches the paper's outline:
        b_i^j is the target of a CCX gate
        controlled by a_i^j and x_i.
    '''
    for j in range(len(a_reg)):
        qc.ccx(a_reg[j], x_i, b_reg[j])


def unbuild_shadow_register(qc: QuantumCircuit, x_i, a_reg, b_reg):
    '''
    CCX is self-inverse, so uncompute by applying the same gates in reverse order.
    '''
    for j in reversed(range(len(a_reg))):
        qc.ccx(a_reg[j], x_i, b_reg[j])

## Explicit controlled-increment adder

This is the adder subroutine used in the notebook.

To add 2^j to a sum register if a control qubit is 1, we perform a controlled increment on the suffix of the sum register starting at bit j.

For example, controlled add 1 to a 3-bit sum s_2 s_1 s_0:

```text
if c == 1:
    s = s + 1 mod 8
```

This is done by toggling high bits when all lower bits carry. It is reversible because it is a circuit of reversible gates. The inverse is the same gates applied in reverse order.

In [ ]:
def apply_gate_tuple(qc, gate_tuple):
    '''
    Apply a stored gate instruction.
    gate_tuple is either:
        ("cx", control, target)
        ("mcx", controls, target)
    '''
    name = gate_tuple[0]
    if name == "cx":
        _, c, t = gate_tuple
        qc.cx(c, t)
    elif name == "mcx":
        _, controls, target = gate_tuple
        if len(controls) == 1:
            qc.cx(controls[0], target)
        else:
            qc.mcx(list(controls), target)
    else:
        raise ValueError(f"Unknown gate tuple: {gate_tuple}")


def controlled_add_power_of_two(qc: QuantumCircuit, control, sum_reg, start_bit: int, inverse=False):
    '''
    Controlled operation:
        if control == 1:
            sum_reg += 2^start_bit  mod 2^W

    sum_reg is little-endian.

    The inverse is obtained by reversing the gate sequence.
    '''
    W = len(sum_reg)
    ops = []

    # Carry propagation from high to low.
    # For k > start_bit:
    # toggle sum[k] if all lower bits sum[start_bit],...,sum[k-1] are 1
    # and the external control is 1.
    for k in range(W - 1, start_bit, -1):
        controls = [control] + [sum_reg[j] for j in range(start_bit, k)]
        ops.append(("mcx", controls, sum_reg[k]))

    # Toggle the starting bit.
    ops.append(("cx", control, sum_reg[start_bit]))

    if inverse:
        ops = list(reversed(ops))

    for op in ops:
        apply_gate_tuple(qc, op)


def add_register_to_sum(qc: QuantumCircuit, addend_reg, sum_reg, inverse=False):
    '''
    Reversible operation:
        |addend>|sum> -> |addend>|sum + addend mod 2^W>

    Implemented by adding 2^j controlled by addend[j].
    '''
    W = len(sum_reg)

    if not inverse:
        bit_range = range(W)
        inv = False
    else:
        bit_range = reversed(range(W))
        inv = True

    for j in bit_range:
        controlled_add_power_of_two(qc, addend_reg[j], sum_reg, j, inverse=inv)


def compute_sum_of_two_registers(qc: QuantumCircuit, left_reg, right_reg, out_reg):
    '''
    Compute:
        out_reg = left_reg + right_reg mod 2^W

    Assumes out_reg starts at |0...0>.
    Keeps left_reg and right_reg unchanged.
    '''
    add_register_to_sum(qc, left_reg, out_reg, inverse=False)
    add_register_to_sum(qc, right_reg, out_reg, inverse=False)


def uncompute_sum_of_two_registers(qc: QuantumCircuit, left_reg, right_reg, out_reg):
    '''
    Undo compute_sum_of_two_registers.
    '''
    add_register_to_sum(qc, right_reg, out_reg, inverse=True)
    add_register_to_sum(qc, left_reg, out_reg, inverse=True)

## Comparison with the target

The paper describes comparing final sum Z with target T.

For a simple baseline, we use the usual method:

1. turn the target bit pattern into the all-ones pattern using X gates,
2. apply a multi-controlled X to a flag qubit,
3. undo the X gates.

This is exactly the kind of large multi-controlled operation that the paper later optimizes away. We keep it here because the goal is a clear baseline oracle.

In [ ]:
def compare_sum_to_target_into_flag(qc: QuantumCircuit, z_reg, target: int, flag):
    '''
    Compute:
        flag ^= 1 iff z_reg == target.

    z_reg is little-endian.
    '''
    target_bits = little_endian_bits(target, len(z_reg))

    # Convert |target> to |111...1>
    for q, bit in zip(z_reg, target_bits):
        if bit == 0:
            qc.x(q)

    # flag ^= AND(all z bits)
    if len(z_reg) == 1:
        qc.cx(z_reg[0], flag)
    else:
        qc.mcx(list(z_reg), flag)

    # Undo conversion
    for q, bit in zip(z_reg, target_bits):
        if bit == 0:
            qc.x(q)


def uncompare_sum_to_target_into_flag(qc: QuantumCircuit, z_reg, target: int, flag):
    '''
    Same circuit again, since the comparison operation is reversible.
    '''
    compare_sum_to_target_into_flag(qc, z_reg, target, flag)

## Oracle body

This is the paper-style oracle body:

```text
build b_i = a_i AND x_i
compute running sums
compare final sum with T into flag
CX(flag, y) for phase kickback
uncompute comparison
uncompute running sums
uncompute b_i
```

The phase-kickback ancilla y must be initialized to |->.

Then CX(flag, y) multiplies the state by -1 exactly when flag = 1.

In [ ]:
def oracle_body(qc, x, y, flag, a_regs, b_regs, sum_regs, S, T):
    '''
    Append one subset-sum oracle body to qc.

    Assumptions:
        - a_regs already hold constants a_i.
        - y is already initialized to |->.
        - b_regs, sum_regs, and flag start at |0>.
    '''

    n = len(S)

    # 1. Build shadow registers b_i = a_i if x_i=1 else 0
    for i in range(n):
        build_shadow_register(qc, x[i], a_regs[i], b_regs[i])

    qc.barrier(label="b_i = a_i AND x_i")

    # 2. Running sums
    if n == 1:
        final_sum = b_regs[0]
    else:
        # s0 = b0 + b1
        compute_sum_of_two_registers(qc, b_regs[0], b_regs[1], sum_regs[0])

        # s1 = s0 + b2, etc.
        for i in range(2, n):
            compute_sum_of_two_registers(qc, sum_regs[i - 2], b_regs[i], sum_regs[i - 1])

        final_sum = sum_regs[n - 2]

    qc.barrier(label="running sums")

    # 3. Compare final sum to target
    compare_sum_to_target_into_flag(qc, final_sum, T, flag[0])

    # 4. Phase kickback via y = |->
    qc.cx(flag[0], y[0])

    # 5. Uncompute comparison
    uncompare_sum_to_target_into_flag(qc, final_sum, T, flag[0])

    qc.barrier(label="comparison + kickback")

    # 6. Uncompute running sums
    if n > 1:
        for i in reversed(range(2, n)):
            uncompute_sum_of_two_registers(qc, sum_regs[i - 2], b_regs[i], sum_regs[i - 1])

        uncompute_sum_of_two_registers(qc, b_regs[0], b_regs[1], sum_regs[0])

    qc.barrier(label="uncompute sums")

    # 7. Uncompute shadow registers
    for i in reversed(range(n)):
        unbuild_shadow_register(qc, x[i], a_regs[i], b_regs[i])

    qc.barrier(label="uncompute shadows")

## Diffusion operator

Grover's diffusion step acts only on the selector register x, not on the arithmetic work registers.

In [ ]:
def diffusion_on_x(qc: QuantumCircuit, x):
    '''
    Standard Grover diffusion operator on x register.
    '''
    qc.h(x)
    qc.x(x)

    if len(x) == 1:
        qc.z(x[0])
    else:
        qc.h(x[-1])
        qc.mcx(list(x[:-1]), x[-1])
        qc.h(x[-1])

    qc.x(x)
    qc.h(x)

## Build the full Grover circuit

For this small example, S={1,2,3}, T=3.

There are two marked subsets among 2^3 = 8 possibilities, so one Grover iteration is enough for a visible amplification.

In [ ]:
# Registers

x = QuantumRegister(n, "x")          # subset selector
y = QuantumRegister(1, "y")          # phase-kickback ancilla
flag = QuantumRegister(1, "flag")    # equality flag

a_regs = [QuantumRegister(W, f"a{i}") for i in range(n)]
b_regs = [QuantumRegister(W, f"b{i}") for i in range(n)]
sum_regs = [QuantumRegister(W, f"s{i}") for i in range(n - 1)]

c = ClassicalRegister(n, "c")

qc = QuantumCircuit(x, y, flag, *a_regs, *b_regs, *sum_regs, c)

# Initialize selector register in uniform superposition
qc.h(x)

# Initialize y to |->
qc.x(y[0])
qc.h(y[0])

# Initialize a_i constant registers
for a_reg, value in zip(a_regs, S):
    initialize_constant_register(qc, a_reg, value)

qc.barrier(label="initialization")

# Choose Grover iteration count.
# This uses the classical solution count only for the demo iteration count,
# not for building the oracle.
M = len(solutions)
N = 2 ** n
num_iterations = max(1, floor((pi / 4) * sqrt(N / M)))

print("Grover iterations:", num_iterations)

for _ in range(num_iterations):
    oracle_body(qc, x, y, flag, a_regs, b_regs, sum_regs, S, T)
    diffusion_on_x(qc, x)
    qc.barrier(label="diffusion")

# Measure only selector bits
for i in range(n):
    qc.measure(x[i], c[i])

print(qc.draw(output="text", fold=-1))

## Simulate

Because Qiskit displays classical bitstrings with the highest-index classical bit on the left, the printed string may look reversed relative to the internal x[0],x[1],x[2] order.

For example, if x[0]=1,x[1]=1,x[2]=0, Qiskit may display the corresponding classical string with c[2] on the left.

The helper below decodes the displayed string back into x-bit order.

In [ ]:
def decode_qiskit_count_key(key: str):
    '''
    Convert a Qiskit count key into x[0], x[1], ..., order.

    Qiskit count strings are usually displayed with the highest classical
    bit on the left. Since we measured x[i] -> c[i], reverse the string.
    '''
    compact = key.replace(" ", "")
    return tuple(int(bit) for bit in compact[::-1])


sim = AerSimulator()
result = sim.run(qc, shots=2000).result()
counts = result.get_counts()

print("Raw Qiskit counts:")
print(counts)

decoded_counts = {}
for key, val in counts.items():
    decoded_counts[decode_qiskit_count_key(key)] = val

print("\nDecoded counts in x[0],x[1],x[2] order:")
print(decoded_counts)

print("\nExpected marked x-bitstrings:")
print(solutions)

## What to check

The high-count outputs should correspond to the marked subsets:

x = (1,1,0), meaning choose 1 and 2,

and

x = (0,0,1), meaning choose 3.

Both satisfy sum_i x_i a_i = 3.

If the counts are noisy or not strongly concentrated, that is normal for such a tiny example with multiple marked states and only one iteration. The key point is that the oracle is constructed from the subset-sum predicate, not from hard-coded solution bitstrings.

## Possible extensions

1. Replace the controlled-increment adder with a cleaner ripple-carry adder.
2. Compare gate counts before and after decomposing `mcx`.
3. Implement the paper's variable-width register idea.
4. Sort S before running sums and measure qubit savings.
5. Replace the large equality-check `mcx` with a CCX-only comparison tree, as discussed in the paper.